In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import gc

print("--- ШАГ 2 (ULTIMATE): ДУАЛЬНЫЙ АНСАМБЛЬ (MAE + RMSE) ---")

# 1. ЗАГРУЗКА
train_df = pd.read_parquet('train_team_track.parquet')
test_df = pd.read_parquet('test_team_track.parquet')

# 2. ВОССТАНАВЛИВАЕМ СВЯЗЬ МАРШРУТ-СКЛАД
print("1/5 Синхронизируем маршруты и склады...")
route_mapping = train_df[['route_id', 'office_from_id']].drop_duplicates()
# Убеждаемся, что в тесте есть office_from_id
if 'office_from_id' not in test_df.columns:
    test_df = test_df.merge(route_mapping, on='route_id', how='left')

# 3. ПОДГОТОВКА ВРЕМЕНИ
for df in [train_df, test_df]:
    df['hour'] = df['timestamp'].dt.hour.astype(np.int8)
    df['dayofweek'] = df['timestamp'].dt.dayofweek.astype(np.int8)
    df['day'] = df['timestamp'].dt.day.astype(np.int8)

# 4. ТРЕНДЫ И ВОЛАТИЛЬНОСТЬ УТРА (10:30)
print("2/5 Вычисляем динамику утра...")
train_sorted = train_df.sort_values(['route_id', 'timestamp'])
# Окно 2 часа (4 замера)
train_sorted['roll_mean'] = train_sorted.groupby('route_id')['target_2h'].transform(lambda x: x.rolling(window=4).mean())
train_sorted['roll_std'] = train_sorted.groupby('route_id')['target_2h'].transform(lambda x: x.rolling(window=4).std())

lag_data = train_sorted[(train_sorted['timestamp'].dt.hour == 10) & (train_sorted['timestamp'].dt.minute == 30)].copy()
info_cols = ['target_2h', 'roll_mean', 'roll_std'] + [f'status_{i}' for i in range(1, 9)]
lag_data = lag_data[['route_id', 'timestamp'] + info_cols]
lag_data.columns = ['route_id', 'lag_ts'] + [f'lag_{c}' for c in info_cols]
lag_data['date'] = lag_data['lag_ts'].dt.date

# 5. СПРАВОЧНИКИ СРЕДНИХ
route_stats = train_df.groupby(['route_id', 'dayofweek', 'hour'])['target_2h'].mean().reset_index()
route_stats.rename(columns={'target_2h': 'hour_mean'}, inplace=True)
office_stats = train_df.groupby('office_from_id')['target_2h'].mean().reset_index()
office_stats.rename(columns={'target_2h': 'office_mean'}, inplace=True)

# 6. ФУНКЦИЯ ПРИЗНАКОВ (БЕЗОПАСНАЯ)
def get_final_features(df, is_test=False):
    df = df.copy()
    df['date'] = df['timestamp'].dt.date

    # Мержим справочники (только те колонки, которых нет)
    df = df.merge(route_stats, on=['route_id', 'dayofweek', 'hour'], how='left')
    df = df.merge(office_stats, on='office_from_id', how='left')

    if is_test:
        current_lag = lag_data[lag_data['lag_ts'] == lag_data['lag_ts'].max()].drop(columns=['lag_ts', 'date'])
        df = df.merge(current_lag, on='route_id', how='left')
        ref_ts = pd.to_datetime('2025-05-30 10:30:00')
    else:
        df = df.merge(lag_data.drop(columns='lag_ts'), on=['route_id', 'date'], how='left')
        df = df.dropna(subset=['lag_target_2h'])
        ref_ts = pd.to_datetime(df['date'].astype(str) + ' 10:30:00')

    df['time_delta'] = (df['timestamp'] - ref_ts).dt.total_seconds() / 3600

    cat_cols = ['route_id', 'office_from_id', 'hour', 'dayofweek']
    num_cols = ['hour_mean', 'office_mean', 'time_delta'] + [f'lag_{c}' for c in info_cols]

    res = df[cat_cols + num_cols].fillna(68.0)
    for c in cat_cols: res[c] = res[c].astype('category')
    return res

# 7. ПОДГОТОВКА ДАТАСЕТОВ
print("3/5 Собираем финальные признаки...")
X_train = get_final_features(train_df)
y_train = train_df.loc[X_train.index, 'target_2h']
X_test = get_final_features(test_df, is_test=True)

# Очистка памяти
del train_df, train_sorted; gc.collect()

# 8. ОБУЧЕНИЕ АНСАМБЛЯ
print("\n4/5 Обучаем специалистов (MAE, RMSE, XGB)...")
cat_features = ['route_id', 'office_from_id', 'hour', 'dayofweek']

# Модель 1: Специалист по WAPE
m_lgb = lgb.LGBMRegressor(objective='mae', n_estimators=1000, learning_rate=0.05, random_state=42).fit(X_train, y_train)
p_mae = np.clip(m_lgb.predict(X_test), 0, None)

# Модель 2: Специалист по Bias
m_cb = CatBoostRegressor(iterations=1000, task_type='GPU', cat_features=cat_features, verbose=0).fit(X_train, y_train)
p_rmse = np.clip(m_cb.predict(X_test), 0, None)

# Модель 3: Балансировщик
m_xgb = xgb.XGBRegressor(tree_method='hist', device='cuda', enable_categorical=True, n_estimators=1000).fit(X_train, y_train)
p_xgb = np.clip(m_xgb.predict(X_test), 0, None)

# 9. СМЕШИВАНИЕ + КАЛИБРОВКА 1.07
print("5/5 Сохраняем сабмит...")
final_preds = (0.4 * p_mae + 0.3 * p_rmse + 0.3 * p_xgb) * 1.07

sub = pd.DataFrame({
    'id': test_df['id'] if 'id' in test_df.columns else test_df.index,
    'y_pred': final_preds
})
sub.to_csv('sub_day2_step2_dual.csv', index=False)
print("\nУРА! Файл 'sub_day2_step2_dual.csv' готов. Загружай и пиши скор!")